# DPSA Vacancy Circular Scraper

Scrapes PDF vacancy links from the South African Department of Public Service and Administration (DPSA) website.

**Features:**
- Scrape a single circular or all circulars listed in the sidebar
- Deduplicates results and filters to PDF links only
- Saves output to CSV

## 1. Imports & Configuration

In [ ]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import pandas as pd
import time

BASE_URL = "https://www.dpsa.gov.za"
CIRCULAR_URL = "https://www.dpsa.gov.za/newsroom/psvc/circular-08-of-2026/"
REQUEST_TIMEOUT = 10  # seconds
DELAY_BETWEEN_REQUESTS = 1  # seconds, used when scraping multiple circulars

## 2. Core Functions

In [ ]:
def fetch_page(url: str) -> BeautifulSoup | None:
    """Fetch a URL and return a BeautifulSoup object, or None on failure."""
    try:
        response = requests.get(url, timeout=REQUEST_TIMEOUT)
        response.raise_for_status()
        return BeautifulSoup(response.text, "html.parser")
    except requests.exceptions.HTTPError as e:
        print(f"HTTP error fetching {url}: {e}")
    except requests.exceptions.ConnectionError:
        print(f"Connection error fetching {url}")
    except requests.exceptions.Timeout:
        print(f"Request timed out for {url}")
    return None


def extract_circular_links(soup: BeautifulSoup, base_url: str) -> list[dict]:
    """
    Extract PDF vacancy links from a circular page.
    Targets only the main content table to avoid duplicates.
    Returns a deduplicated list of {title, url} dicts.
    """
    # Target the main content div to avoid picking up sidebar/footer duplicates
    content = soup.find(id="content")
    if not content:
        print("Warning: could not find #content element, falling back to full page")
        content = soup

    seen_urls = set()
    links = []

    for anchor in content.find_all("a", href=True):
        href = anchor["href"]
        title = anchor.get_text(strip=True)

        # Filter to PDF links only
        if not href.lower().endswith(".pdf"):
            continue

        absolute_url = urljoin(base_url, href)

        # Deduplicate
        if absolute_url in seen_urls:
            continue
        seen_urls.add(absolute_url)

        links.append({"title": title, "url": absolute_url})

    return links


def get_all_circular_urls(soup: BeautifulSoup, base_url: str) -> list[str]:
    """
    Extract all circular page URLs from the sidebar navigation.
    Returns a list of absolute URLs.
    """
    sidebar = soup.find(id="sidebar")
    if not sidebar:
        print("Warning: could not find sidebar")
        return []

    urls = []
    for anchor in sidebar.find_all("a", href=True):
        href = anchor["href"]
        if "/newsroom/psvc/circular-" in href:
            urls.append(urljoin(base_url, href))

    return urls


def save_to_csv(links: list[dict], filename: str) -> None:
    """Save a list of link dicts to a CSV file."""
    df = pd.DataFrame(links)
    df.to_csv(filename, index=False)
    print(f"Saved {len(df)} links to {filename}")

## 3. Scrape a Single Circular

In [ ]:
soup = fetch_page(CIRCULAR_URL)

if soup:
    links = extract_circular_links(soup, BASE_URL)
    df_single = pd.DataFrame(links)
    print(f"Found {len(links)} PDF links\n")
    print(df_single.to_string(index=False))
else:
    print("Failed to fetch page.")

In [ ]:
save_to_csv(links, "circular_08_2026_links.csv")

## 4. (Optional) Scrape All Circulars

Loops over every circular listed in the sidebar and collects all PDF links into one CSV.
A short delay is added between requests to be polite to the server.

In [ ]:
if soup:
    circular_urls = get_all_circular_urls(soup, BASE_URL)
    print(f"Found {len(circular_urls)} circulars to scrape")

    all_links = []

    for i, circular_url in enumerate(circular_urls, start=1):
        circular_name = circular_url.rstrip("/").split("/")[-1]
        print(f"[{i}/{len(circular_urls)}] Scraping {circular_name}...")

        page = fetch_page(circular_url)
        if page:
            page_links = extract_circular_links(page, BASE_URL)
            # Tag each link with its source circular
            for link in page_links:
                link["circular"] = circular_name
            all_links.extend(page_links)
        else:
            print(f"  Skipping {circular_name} — failed to fetch")

        time.sleep(DELAY_BETWEEN_REQUESTS)

    print(f"\nTotal links collected: {len(all_links)}")
    save_to_csv(all_links, "all_circulars_links.csv")